In [ ]:
import gc
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import nltk

from collections import defaultdict
from tqdm import tqdm

# scikit-learn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2, mutual_info_classif, VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    f1_score, precision_score, recall_score,
    average_precision_score,
)
from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV,
    TunedThresholdClassifierCV,
)
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder, OrdinalEncoder, OneHotEncoder,
)

# imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline

# HuggingFace
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset

# XGBoost
from xgboost import XGBClassifier

# NLTK
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

nltk.download("stopwords", quiet=True)

warnings.filterwarnings("ignore")

# Free GPU memory if restarting mid-session
gc.collect()
torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 2. Configuration & Utilities

In [ ]:
# ── Column definitions ──────────────────────────────────────────────────────
TEXT_COL   = "prognosis_english"
LABEL_COL  = "is_effective_romexis"
DATA_PATH  = "data/Erda, final, anonymized,for LLM and multimodal.csv"

ONEHOT_COLS = [
    "gender", "tooth_type", "diagnosis_grouped", "side_correct", "jaw",
    "smoke", "diabetes", "diseases_immune_system", "heart_disease",
    "pre_treatment_factors_binary_updated",
    "treatment_factors_binary_updated_final",
    "posttreatment_factors_binary_updated",
    "crown_status",
]

AGE_BUCKET_CLASSES = ["15-24", "25-34", "35-44", "45-54", "55-64", "≥65"]

FOLLOW_UP_CLASSES = [
    "< 3 months", "3-6 months", "6-12 months", "12-24 months", "> 24 months",
]

TAB_COLS = ONEHOT_COLS + ["age_bucket", "follow_up_category_romexis"]

In [ ]:
def preprocess_text(text: str, language: str = "english") -> str:
    """Remove stopwords and punctuation, then apply Snowball stemming."""
    if str(text) == "nan":
        return ""
    text = str(text).replace("_x000D_", "\n")
    stemmer    = SnowballStemmer(language)
    stop_words = set(stopwords.words(language))
    text       = text.translate(str.maketrans("", "", string.punctuation))
    tokens     = [w for w in text.split() if w.lower() not in stop_words]
    return " ".join(stemmer.stem(w) for w in tokens)


def nested_cv(
    pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    param_grid: dict,
    n_split_inner: int = 5,
    n_split_outer: int = 5,
    n_trials: int = 5,
    scoring: str = "roc_auc",
    threshold_optimization: bool = True,
) -> dict:
    """
    Repeated nested cross-validation with optional threshold optimisation.

    Outer loop: performance estimation.
    Inner loop: hyperparameter selection via GridSearchCV.
    Threshold loop (optional): TunedThresholdClassifierCV on the training fold.

    Returns a dict mapping metric name → (mean, std) over all outer folds × trials.
    """
    score_acc = defaultdict(list)

    for trial in tqdm(range(n_trials), total=n_trials, desc="Trials"):
        outer_cv = StratifiedKFold(n_splits=n_split_outer, random_state=trial, shuffle=True)
        inner_cv = StratifiedKFold(n_splits=n_split_inner, random_state=trial, shuffle=True)

        fold_roc_auc, fold_f1, fold_prec, fold_ap, fold_rec = [], [], [], [], []

        for train_idx, test_idx in outer_cv.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            gs = GridSearchCV(
                estimator=pipeline, param_grid=param_grid,
                cv=inner_cv, scoring=scoring, n_jobs=-1
            ).fit(X_train, y_train)

            if threshold_optimization:
                best_clf = TunedThresholdClassifierCV(
                    gs.best_estimator_, cv=10, refit=True,
                    scoring=scoring, n_jobs=-1
                ).fit(X_train, y_train)
            else:
                best_clf = gs.best_estimator_

            y_pred       = best_clf.predict(X_test)
            y_pred_proba = best_clf.predict_proba(X_test)[:, 1]

            fold_roc_auc.append(roc_auc_score(y_test, y_pred_proba))
            fold_f1.append(f1_score(y_test, y_pred, average="macro"))
            fold_prec.append(precision_score(y_test, y_pred, zero_division=0))
            fold_ap.append(average_precision_score(y_test, y_pred_proba))
            fold_rec.append(recall_score(y_test, y_pred))

        score_acc["roc_auc"].append(np.mean(fold_roc_auc))
        score_acc["average_precision"].append(np.mean(fold_ap))
        score_acc["f1_macro"].append(np.mean(fold_f1))
        score_acc["precision"].append(np.mean(fold_prec))
        score_acc["recall"].append(np.mean(fold_rec))

    return {
        k: (round(np.mean(v), 4), round(np.std(v), 4))
        for k, v in score_acc.items()
    }


def get_oof_proba(
    pipeline,
    param_grid: dict,
    X: pd.DataFrame,
    y: pd.Series,
    n_split_inner: int = 5,
    n_split_outer: int = 5,
    scoring: str = "roc_auc",
    random_state: int = 42,
):
    """
    Collect out-of-fold predicted probabilities via nested CV.
    Used for pooled ROC curve construction.
    """
    outer_cv = StratifiedKFold(n_splits=n_split_outer, shuffle=True, random_state=random_state)
    inner_cv = StratifiedKFold(n_splits=n_split_inner, shuffle=True, random_state=random_state)

    y_true_all, y_score_all = [], []

    for train_idx, test_idx in outer_cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        gs = GridSearchCV(
            pipeline, param_grid=param_grid,
            cv=inner_cv, scoring=scoring, n_jobs=-1, refit=True
        ).fit(X_train, y_train)

        y_score_all.append(gs.best_estimator_.predict_proba(X_test)[:, 1])
        y_true_all.append(y_test.to_numpy())

    return np.concatenate(y_true_all), np.concatenate(y_score_all)

## 3. Data Loading & Cleaning

Standardises `age_bucket` and `follow_up_category_romexis` labels, which contain
encoding inconsistencies in the source file.

In [ ]:
df = pd.read_csv(DATA_PATH, sep=None, engine="python")

# ── Standardise age_bucket ───────────────────────────────────────────────
df["age_bucket"] = (
    df["age_bucket"].fillna("missing").astype(str).str.strip()
    .replace({"more than 65": "≥65", "65+": "≥65", "65 +": "≥65",
              ">=65": "≥65", ">65": "≥65"})
)

# ── Standardise follow_up_category_romexis ───────────────────────────────
df["follow_up_category_romexis"] = (
    df["follow_up_category_romexis"]
    .fillna("missing").astype(str).str.strip()
    .str.replace("–", "-", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .replace({"3- 6 months": "3-6 months",
              "6- 12 months": "6-12 months",
              "12- 24 months": "12-24 months"})
)

print("Shape:", df.shape)
print("Label distribution:\n", df[LABEL_COL].value_counts())

## 4. Feature Engineering

### 4a. Tabular features

Nominal variables are one-hot encoded; ordinal variables (`age_bucket`,
`follow_up_category_romexis`) use ordered ordinal encoding.

In [ ]:
X_tab = df[TAB_COLS].fillna("missing").astype(str).copy()
y     = df[LABEL_COL]

preprocessor = ColumnTransformer(
    transformers=[
        ("onehot",
         OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
         ONEHOT_COLS),

        ("age_ord",
         OrdinalEncoder(
             categories=[AGE_BUCKET_CLASSES + ["missing"]],
             handle_unknown="use_encoded_value",
             unknown_value=-1,
         ),
         ["age_bucket"]),

        ("follow_up_ord",
         OrdinalEncoder(
             categories=[FOLLOW_UP_CLASSES + ["missing"]],
             handle_unknown="use_encoded_value",
             unknown_value=-1,
         ),
         ["follow_up_category_romexis"]),
    ],
    remainder="drop",
)

X_tab_enc = preprocessor.fit_transform(X_tab)
print("Tabular feature matrix:", X_tab_enc.shape)

### 4b. Chi² feature importance

Quick sanity check on the encoded tabular features before combining with text embeddings.

In [ ]:
chi2_scores, p_values = chi2(X_tab_enc, y)

feature_names = np.array(
    preprocessor.named_transformers_["onehot"].get_feature_names_out(ONEHOT_COLS).tolist()
    + ["age_bucket", "follow_up_category_romexis"]
)

top_idx = np.argsort(chi2_scores)[::-1][:14]

plt.figure(figsize=(10, 6))
plt.barh(feature_names[top_idx][::-1], chi2_scores[top_idx][::-1])
plt.xlabel("Chi² Score")
plt.title("Top 14 Features by Chi² Test")
plt.tight_layout()
plt.show()

## 5. Text Embeddings

The prognosis text is tokenised with DistilBERT's subword tokeniser and passed through
the pretrained encoder. Sentence-level representations are obtained via
attention-mask-aware mean pooling over the final hidden states.

In [ ]:
class TokenEmbeddings(BaseEstimator, TransformerMixin):
    """
    sklearn-compatible transformer that produces mean-pooled DistilBERT embeddings.

    Args:
        model_name: HuggingFace model identifier.
        device:     'cuda' or 'cpu'.
        batch_size: number of examples per inference batch.
    """
    def __init__(self, model_name: str = "distilbert-base-uncased",
                 device: str = "cpu", batch_size: int = 32):
        self.model_name = model_name
        self.device     = device
        self.batch_size = batch_size

    def fit(self, X, y=None):
        self.model_ = AutoModel.from_pretrained(self.model_name).to(self.device)
        self.model_.eval()
        return self

    @staticmethod
    def _mean_pool(last_hidden_state, attention_mask):
        mask   = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    def transform(self, X: dict) -> np.ndarray:
        """
        X must be a dict with keys 'input_ids' and 'attention_mask'
        (as returned by a HuggingFace tokenizer).
        """
        input_ids      = X["input_ids"]
        attention_mask = X["attention_mask"]
        emb_list       = []

        with torch.no_grad():
            for i in range(0, len(input_ids), self.batch_size):
                ids  = torch.tensor(input_ids[i: i + self.batch_size]).to(self.device)
                mask = torch.tensor(attention_mask[i: i + self.batch_size]).to(self.device)
                out  = self.model_(input_ids=ids, attention_mask=mask)
                emb_list.append(self._mean_pool(out.last_hidden_state, mask).cpu().numpy())

        return np.vstack(emb_list)

In [ ]:
# Tokenise the prognosis text column
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_texts(texts, tokenizer, max_length: int = 128):
    return tokenizer(
        [str(t) for t in texts],
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )

token_dict = tokenize_texts(df[TEXT_COL].tolist(), tokenizer)

# Extract embeddings
text_encoder   = TokenEmbeddings(model_name="distilbert-base-uncased",
                                  device=DEVICE, batch_size=32)
text_embeddings = text_encoder.fit_transform(token_dict)
print("Text embedding matrix:", text_embeddings.shape)

## 6. Combined Feature Matrix

In [ ]:
X_final    = np.hstack([text_embeddings, X_tab_enc])
X_final_df = pd.DataFrame(X_final, index=df.index)

print("Text embeddings : ", text_embeddings.shape)
print("Tabular features: ", X_tab_enc.shape)
print("Combined matrix : ", X_final_df.shape)

## 7. Model Evaluation — Nested CV

All models are evaluated using repeated nested cross-validation
(5 outer folds × 5 inner folds × 5 trials) with threshold optimisation on each training fold.
Reported metrics: ROC-AUC, PR-AUC, macro-F1, precision, recall (mean ± std).

In [ ]:
models_to_run = {
    "LogisticRegression": (
        ImbPipeline([
            ("variance_threshold", VarianceThreshold(0)),
            ("classifier", LogisticRegression(max_iter=500, random_state=42)),
        ]),
        {
            "classifier__C":      [0.0001, 0.01, 0.1, 1],
            "classifier__solver": ["liblinear"],
        },
    ),
    "GaussianNB": (
        ImbPipeline([
            ("variance_threshold", VarianceThreshold(0)),
            ("classifier", GaussianNB()),
        ]),
        {},
    ),
    "RandomForest": (
        ImbPipeline([
            ("variance_threshold", VarianceThreshold(0)),
            ("classifier", RandomForestClassifier(n_estimators=200, random_state=42)),
        ]),
        {
            "classifier__max_depth":         [None, 10, 20],
            "classifier__min_samples_split": [2, 5, 10],
            "classifier__min_samples_leaf":  [1, 2, 4],
        },
    ),
    "XGBoost": (
        ImbPipeline([
            ("variance_threshold", VarianceThreshold(0)),
            ("classifier", XGBClassifier(eval_metric="logloss", random_state=42)),
        ]),
        {
            "classifier__max_depth":     [3, 5, 7],
            "classifier__learning_rate": [0.01, 0.1, 0.2],
            "classifier__n_estimators":  [100, 200],
        },
    ),
}

In [ ]:
results = {}

for name, (pipeline, param_grid) in models_to_run.items():
    print(f"\nRunning nested CV: {name}")
    results[name] = nested_cv(
        pipeline=pipeline,
        X=X_final_df,
        y=y,
        param_grid=param_grid,
        n_split_inner=5,
        n_split_outer=5,
        n_trials=5,
        threshold_optimization=True,
    )
    print(f"  {results[name]}")

print("\n=== Summary ===")
for name, scores in results.items():
    print(f"  {name}: {scores}")

## 8. ROC Curve Visualisation

Out-of-fold predicted probabilities are pooled across all outer folds to construct
a single ROC curve per model.

In [ ]:
colors = plt.cm.Set2.colors

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

for i, (model_name, (pipeline, param_grid)) in enumerate(models_to_run.items()):
    print(f"Generating OOF probabilities: {model_name}")
    y_true, y_score = get_oof_proba(
        pipeline=pipeline,
        param_grid=param_grid,
        X=X_final_df,
        y=y,
        n_split_inner=5,
        n_split_outer=5,
        random_state=42,
    )
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2.5,
            color=colors[i % len(colors)],
            label=f"{model_name} (AUC = {roc_auc:.2f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="ROC Curves (Nested CV, pooled outer folds)",
       xlim=(0, 1), ylim=(0, 1))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig("ROC_ML_token_embeddings.png", dpi=600, bbox_inches="tight")
plt.show()